# q2: traffic sources
which channels bring visitors who buy?
channels are grouped from session source / medium (q2b in sql/04_traffic_sources.sql).

In [1]:
import warnings
warnings.filterwarnings("ignore", message="BigQuery Storage module not found")

from pathlib import Path
from google.cloud import bigquery

client = bigquery.Client(project="ga4-funnel-analysis-511719")
output_dir = Path("../outputs")
output_dir.mkdir(exist_ok=True)

In [2]:
sql = """
select
    case
        when medium is null then 'unknown'
        when source in ('shop.googlemerchandisestore.com', 'googlemerchandisestore.com') then 'self-referral'
        when source = '(direct)' then 'direct'
        when medium = 'organic' then 'organic search'
        when medium = 'cpc' then 'paid search'
        when medium = 'referral' then 'referral'
        when medium = 'email' then 'email'
        when medium = 'affiliate' then 'affiliate'
        when medium in ('<Other>', '(data deleted)') then 'hidden'
        else 'other'
    end as channel,
    count(*) as sessions,
    countif(viewed_item or added_to_cart or began_checkout or purchased) as reached_view,
    countif(purchased) as purchases,
    sum(revenue) as revenue
from `ga4-funnel-analysis-511719.ga4_analysis.session_base`
group by channel
order by sessions desc
"""

channels = client.query(sql).to_dataframe()
channels

,channel,sessions,reached_view,purchases,revenue
0,organic search,101888,26645,1364,95902.0
1,unknown,94554,6434,1,12.0
2,self-referral,57590,18162,1194,100254.0
3,direct,37839,8647,255,21237.0
4,hidden,31771,7170,267,18780.0
5,referral,27289,8293,899,66669.0
6,paid search,7672,1658,42,2960.0
7,affiliate,1329,176,2,167.0
8,email,197,129,20,1659.0


In [3]:
channels["view_rate"] = (channels["reached_view"] / channels["sessions"]).round(4)
channels["conversion_rate"] = (channels["purchases"] / channels["sessions"]).round(4)
channels["revenue_per_session"] = (channels["revenue"] / channels["sessions"]).round(2)
print(channels["sessions"].sum(), channels["purchases"].sum(), channels["revenue"].sum())
channels

360129 4044 307640.0


,channel,sessions,reached_view,purchases,revenue,view_rate,conversion_rate,revenue_per_session
0,organic search,101888,26645,1364,95902.0,0.2615,0.0134,0.94
1,unknown,94554,6434,1,12.0,0.068,0.0,0.0
2,self-referral,57590,18162,1194,100254.0,0.3154,0.0207,1.74
3,direct,37839,8647,255,21237.0,0.2285,0.0067,0.56
4,hidden,31771,7170,267,18780.0,0.2257,0.0084,0.59
5,referral,27289,8293,899,66669.0,0.3039,0.0329,2.44
6,paid search,7672,1658,42,2960.0,0.2161,0.0055,0.39
7,affiliate,1329,176,2,167.0,0.1324,0.0015,0.13
8,email,197,129,20,1659.0,0.6548,0.1015,8.42


In [4]:
channels.to_csv(output_dir / "q2_channels.csv", index=False)